In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
import subprocess
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/vlm-finetuning-project1"
REPO_DIR = "vlm-safety-reasoning"
ENV_PATH = f"{DRIVE_ROOT}/secrets/.env"

def load_secrets(env_path: str) -> dict:
    """Read a .env file and export its values into os.environ."""
    if not os.path.exists(env_path):
        raise FileNotFoundError(f"Secrets file not found at: {env_path}")

    secrets = {}
    with open(env_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            secrets[key] = value.strip(" \"'\r")
            os.environ[key] = secrets[key]
    return secrets


print(">>> Loading secrets...")
secrets = load_secrets(ENV_PATH)
required_keys = ["GIT_EMAIL", "GIT_NAME", "GITHUB_USERNAME", "GITHUB_TOKEN", "HF_TOKEN"]
missing = [k for k in required_keys if k not in secrets]
if missing:
    raise KeyError(f"Missing required secrets: {missing}")
print(">>> Secrets loaded successfully.")

print(">>> Configuring Git identity...")
subprocess.run(["git", "config", "--global", "user.email", secrets["GIT_EMAIL"]], check=True)
subprocess.run(["git", "config", "--global", "user.name", secrets["GIT_NAME"]], check=True)

AUTH_REPO_URL = (
    f"https://{secrets['GITHUB_USERNAME']}:{secrets['GITHUB_TOKEN']}"
    f"@github.com/epmresearch/vlm-safety-reasoning.git"
)

if os.path.exists(REPO_DIR):
    print(">>> Repo already present, pulling latest...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "remote", "set-url", "origin", AUTH_REPO_URL], check=True)
    subprocess.run(["git", "pull", "origin", "main"], check=True)
else:
    print(">>> Cloning repo...")
    subprocess.run(["git", "clone", AUTH_REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print(f">>> Working directory: {os.getcwd()}")

print(">>> Copying .env into local workspace...")
shutil.copy(ENV_PATH, ".env")

print(">>> Installing requirements...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(">>> Setup complete.")

>>> Loading secrets...
>>> Secrets loaded successfully.
>>> Configuring Git identity...
>>> Repo already present, pulling latest...
>>> Working directory: /content/vlm-safety-reasoning
>>> Copying .env into local workspace...
>>> Installing requirements...
>>> Setup complete.


In [5]:
from core.config import load_config
from data.loader import load_processed_dataset
from data.preprocessor import raw_sample_to_conversation  # note: per-sample fn, not the batch builder

sft_cfg = load_config(training_kind="sft")
print("Configured max_seq_length:", sft_cfg.get("max_seq_length"))
print("image_min_pixels:", sft_cfg.get("image_min_pixels"))
print("image_max_pixels:", sft_cfg.get("image_max_pixels"))

from unsloth import FastVisionModel
from models.model_loader import get_model_info, apply_pixel_bounds

MODEL_TIER = "2b"
model_name = get_model_info(MODEL_TIER)["hf_path"]

model, tokenizer = FastVisionModel.from_pretrained(
    model_name,
    load_in_4bit=True,
    max_seq_length=sft_cfg.get("max_seq_length", 4096),
)
apply_pixel_bounds(
    tokenizer,
    min_pixels=sft_cfg.get("image_min_pixels"),
    max_pixels=sft_cfg.get("image_max_pixels"),
)
print("Pixel bounds applied — tokenizer ready.")

Configured max_seq_length: 4096
image_min_pixels: 200704
image_max_pixels: 1204224
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

2026-07-20 07:07:34 | INFO     | models.model_loader:apply_pixel_bounds:232 - Applied image pixel bounds via size dict: min=200704, max=1204224
Pixel bounds applied — tokenizer ready.


In [6]:
splits = load_processed_dataset()
train_raw = splits["train"]
val_raw = splits["val"]
test_raw = splits["test"]
print(f"Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")

2026-07-20 07:07:46 | INFO     | data.loader:load_processed_dataset:183 - Loading fully processed dataset from disk: /content/drive/MyDrive/vlm-finetuning-project1/datasets/processed
2026-07-20 07:07:47 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'train' split: 6308 samples
2026-07-20 07:07:47 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'val' split: 701 samples
2026-07-20 07:07:47 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'test' split: 3004 samples
Train: 6308  Val: 701  Test: 3004


In [7]:
from qwen_vl_utils import process_vision_info
import numpy as np
from tqdm.auto import tqdm
import gc

def get_full_sequence_length_from_raw(raw_sample, tokenizer):
    """Builds ONE conversation, measures it, lets it go out of scope immediately."""
    conv = raw_sample_to_conversation(raw_sample, raw_sample["image"])
    messages = conv["messages"]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=False, tokenize=False)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = tokenizer(text=text, images=image_inputs, videos=video_inputs, return_tensors="pt")
    return inputs["input_ids"].shape[1]

def get_prompt_only_length_from_raw(raw_sample, tokenizer):
    """Same, but only system+user (no assistant target) — inference-style."""
    conv = raw_sample_to_conversation(raw_sample, raw_sample["image"])
    messages = conv["messages"][:2]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = tokenizer(text=text, images=image_inputs, videos=video_inputs, return_tensors="pt")
    return inputs["input_ids"].shape[1]

def sweep_split(raw_dataset, tokenizer, mode, label):
    """mode: 'full' (train-style) or 'prompt_only' (inference-style).
    Iterates the HF dataset directly — never builds a full in-memory list of conversations."""
    fn = get_full_sequence_length_from_raw if mode == "full" else get_prompt_only_length_from_raw
    lens, fails = [], []
    for idx in tqdm(range(len(raw_dataset)), desc=f"{label} [{mode}]"):
        try:
            lens.append(fn(raw_dataset[idx], tokenizer))
        except Exception as e:
            fails.append((idx, str(e)))
    return np.array(lens), fails

def print_stats(arr, label, limit=None):
    print(f"\n=== {label} ===")
    print(f"Min:{arr.min()}  Mean:{arr.mean():.0f}  Median:{np.median(arr):.0f}  "
          f"P90:{np.percentile(arr,90):.0f}  P95:{np.percentile(arr,95):.0f}  "
          f"P99:{np.percentile(arr,99):.0f}  Max:{arr.max()}")
    if limit:
        over = (arr > limit).sum()
        print(f"Exceeding limit={limit}: {over} / {len(arr)}")

In [8]:
train_full_arr, train_full_fails = sweep_split(train_raw, tokenizer, "full", "TRAIN")
print(f"Failures: {len(train_full_fails)}")
print_stats(train_full_arr, "TRAIN full-sequence (system+user+target)", limit=sft_cfg.get("max_seq_length", 4096))

recommended_train = int(np.percentile(train_full_arr, 99.5) * 1.1)
print(f"\nSuggested training max_seq_length (P99.5 + 10% margin): {recommended_train}")

# free memory before next split
del train_full_arr
gc.collect()

TRAIN [full]:   0%|          | 0/6308 [00:00<?, ?it/s]

Failures: 0

=== TRAIN full-sequence (system+user+target) ===
Min:647  Mean:1449  Median:1535  P90:1604  P95:1627  P99:1676  Max:1865
Exceeding limit=4096: 0 / 6308

Suggested training max_seq_length (P99.5 + 10% margin): 1868


151582

In [9]:
val_full_arr, val_full_fails = sweep_split(val_raw, tokenizer, "full", "VAL")
print(f"Failures: {len(val_full_fails)}")
print_stats(val_full_arr, "VAL full-sequence (system+user+target)", limit=sft_cfg.get("max_seq_length", 4096))

del val_full_arr
gc.collect()

VAL [full]:   0%|          | 0/701 [00:00<?, ?it/s]

Failures: 0

=== VAL full-sequence (system+user+target) ===
Min:723  Mean:1446  Median:1533  P90:1605  P95:1623  P99:1667  Max:1728
Exceeding limit=4096: 0 / 701


66113

In [10]:
test_prompt_arr, test_prompt_fails = sweep_split(test_raw, tokenizer, "prompt_only", "TEST")
print(f"Failures: {len(test_prompt_fails)}")
print_stats(test_prompt_arr, "TEST prompt-only (system+user+image, no target)")

MAX_NEW_TOKENS = 1000
needed = test_prompt_arr.max() + MAX_NEW_TOKENS
print(f"\nWorst-case inference need: {test_prompt_arr.max()} + {MAX_NEW_TOKENS} = {needed}")
print(f"With 10% margin: {int(needed * 1.1)}")

del test_prompt_arr
gc.collect()

TEST [prompt_only]:   0%|          | 0/3004 [00:00<?, ?it/s]

Failures: 0

=== TEST prompt-only (system+user+image, no target) ===
Min:543  Mean:1325  Median:1407  P90:1407  P95:1474  P99:1474  Max:1519

Worst-case inference need: 1519 + 1000 = 2519
With 10% margin: 2770


122417